# Carro RC con ESP32: funcionamiento del proyecto

Documentación técnica del circuito, el firmware y el proceso de puesta en marcha de un carro a control remoto con tracción por motor DC y dirección por servo, controlado por una ESP32. Proyecto personal en desarrollo desde 2023.

**Objetivo final:** manejar el carro con un control físico (Xbox) para pruebas de carrera, donde la precisión de la dirección es el requisito principal.

| Estado | Detalle |
|---|---|
| Motor de tracción | Validado en banco con LiPo + LM2596 + BTS7960 |
| Dirección (servo en D21) | Validada con rutina automática de prueba |
| Control Xbox | Se empareja y los joysticks se leen correctamente |
| **Problema abierto** | Al mover la dirección con el control, este se desconecta (ver sección 9) |

## Contenido

1. Materiales
2. Arquitectura de alimentación
3. Conexiones
4. Ajuste del regulador LM2596
5. Tierra común
6. Firmware de prueba (servo + motor)
7. Firmware con control Xbox
8. Procedimiento de puesta en marcha
9. Estado actual y problema abierto
10. Historial de pruebas y lecciones aprendidas
11. Seguridad
12. Pendientes

## 1. Materiales

| Componente | Función | Notas |
|---|---|---|
| ESP32 + placa de expansión | Controlador | Puerto DC de 6.5 a 16 V, USB-C y micro-USB, jumper 5V/3.3V para la fila V |
| BTS7960 | Puente H del motor | Lógica a 5 V. Alimentación de potencia (B+/B−) de 5.5 a 27 V |
| Motor RS550 de 6 V | Tracción | Alto consumo de arranque |
| Servo MG995 | Dirección | Pulso de 500 a 2400 µs |
| LM2596 (ajustable) | Regulador de 11.1 V a 6 V para el motor | Corriente máxima aproximada de 3 A |
| LiPo 11.1 V, 460 mAh, 35C | Fuente del motor | Capacidad de descarga de unos 16 A |
| Pack de 3 celdas 18650 | Fuente de la ESP32 y el servo | Entra por el puerto DC de la placa |
| Pack de 2 celdas 18650 en serie (7.4 V) | Fuente del motor en las primeras pruebas | Ya no se usa en el montaje final |
| Interruptores SW1 y SW2 | Encendido de cada fuente | SW2 debe soportar 10 A o más |
| Fusible de 10 a 15 A | Protección de la LiPo | En el positivo de la LiPo |
| Control inalámbrico Xbox | Mando | Identificado por Bluepad32 como "XBox One", VID 0x045E, PID 0x02FD, firmware 5.23.6.0 |

Descartados en esta revisión: módulo HC-06 y ESP8266. El control ahora va directo por Bluetooth de la ESP32.

> 📷 **Imagen pendiente:** `img/materiales.png` — Fotografía de todos los componentes (archivo aún no disponible)

## 2. Arquitectura de alimentación

El circuito usa **dos fuentes separadas con tierra común**. Así los picos de corriente del motor no afectan a la electrónica de control.

| Fuente | Alimenta | Recorrido |
|---|---|---|
| Pack de 3 × 18650 (11.1 V) | ESP32, placa de expansión y servo | SW1, puerto DC de la placa |
| LiPo 11.1 V | Motor RS550 | Fusible, SW2, **LM2596 a 6 V**, bornes B+/B− del BTS7960 |

Reglas deducidas de las pruebas:

- El **interruptor de la LiPo va antes de IN+ del LM2596**. Si va después de OUT+, el regulador queda siempre encendido y descarga la batería aunque el circuito parezca apagado.
- El BTS7960 se **bloquea por debajo de 5.5 V** en B+/B−. Con una sola celda de 3.7 V el motor no gira.
- Mientras la placa se alimenta por el puerto DC, no se debe conectar el USB al mismo tiempo. No se verificó si la placa aísla ambas fuentes.

> **Insertar imagen aquí:** `img/diagrama_alimentacion.png` — Diagrama de alimentación del carro
![Diagrama de alimentación del carro](img/diagrama_alimentacion.png)

## 3. Conexiones

### 3.1 BTS7960 a la placa de expansión

| Pin del BTS7960 | Pin de la placa |
|---|---|
| A (GND) | GND |
| B (VCC) | 5V |
| E (L_EN) y F (R_EN), unidos con un puente corto | D22 |
| G (LPWM, reversa) | D18 |
| H (RPWM, adelante) | D5 |
| C y D | Sin conectar |

> 📷 **Imagen pendiente:** `img/bts7960_pinout.png` — BTS7960 y distribución de pines del bloque de 8 (archivo aún no disponible)

### 3.2 Bornes de potencia del BTS7960

| Borne | Conexión |
|---|---|
| 1 (B−) | OUT− del LM2596 |
| 2 (B+) | OUT+ del LM2596 (6 V) |
| 3 (M+) | Cable rojo del motor |
| 4 (M−) | Cable negro del motor |

### 3.3 Servo MG995

| Cable | Pin de la placa |
|---|---|
| Naranja (señal) | D21 |
| Rojo | 5V |
| Café | GND |

### 3.4 Regulador y fuentes

| Desde | Hacia |
|---|---|
| LiPo (+) | Fusible, SW2, IN+ del LM2596 |
| LiPo (−) | IN− del LM2596 |
| Pack de 3 celdas | Puerto DC de la placa, con SW1 en el positivo |

### 3.5 Jumper de la placa de expansión

El jumper amarillo junto al USB-C elige el voltaje de la fila V:

| Posición | Voltaje en la fila V |
|---|---|
| Cubre 5V y V (los dos pines de arriba) | **5 V (correcto para este proyecto)** |
| Cubre V y 3.3V (los dos de abajo) | 3.3 V |

> 📷 **Imagen pendiente:** `img/placa_expansion.png` — Placa de expansión con el jumper en 5V (archivo aún no disponible)

> **Insertar imagen aquí:** `img/diagrama_senales_colores_pines.png` — Diagrama de señales y pines con un color por pin
![Diagrama de señales y pines con un color por pin](img/diagrama_senales_colores_pines.png)

> 📷 **Imagen pendiente:** `img/montaje_real.png` — Fotografía del montaje real del carro (archivo aún no disponible)

## 4. Ajuste del regulador LM2596

El regulador se calibra **antes** de conectarlo al BTS7960.

1. Conectar solo la LiPo (con su interruptor y fusible) a IN+ e IN−.
2. Con el multímetro en OUT+ y OUT−, girar despacio el tornillo azul hasta leer **6.0 V**.
3. Apagar y conectar OUT+ al borne 2 y OUT− al borne 1 del BTS7960.

En el firmware, la constante `VOLTAJE_BATERIA` debe coincidir con este voltaje (6.0). Si el motor se alimenta de otra fuente, se actualiza el valor y el código recalcula el tope de PWM para no pasar de 6 V efectivos en el motor.

| Fuente del motor | `VOLTAJE_BATERIA` | Tope de PWM |
|---|---|---|
| LM2596 a 6 V | 6.0 | 255 |
| 2 celdas 18650 en serie | 7.4 | 206 |
| LiPo directa, 12.6 V | 12.6 | 121 |

**Limitación:** el LM2596 entrega unos 3 A. Con carga pesada, el voltaje puede caer por debajo de 5.5 V y el BTS7960 se bloquea. Esto todavía no se ha validado con el motor trabajando bajo carga.

> 📷 **Imagen pendiente:** `img/lm2596.png` — Regulador LM2596 con el tornillo de ajuste (archivo aún no disponible)

## 5. Tierra común

Las señales de la ESP32 solo se interpretan respecto a su propio GND, así que el BTS7960 debe compartirlo. Se resuelve con **un solo cable** del pin A (GND) del BTS7960 a un GND de la placa:

1. El GND de la placa es el negativo del pack de 3 celdas.
2. El pin A del BTS7960 está unido internamente al borne 1 (B−). Se comprobó con continuidad.
3. El borne 1 va a OUT− del LM2596.
4. En estos reguladores, IN− y OUT− son el mismo punto, y IN− va al negativo de la LiPo.

Todos los negativos quedan unidos. No hace falta un segundo cable ni tomar la tierra antes del regulador.

**Comprobación con el multímetro en continuidad, todo apagado:**

| Revisión | Resultado esperado |
|---|---|
| Pin A y borne 1 | Suena |
| Negativo de la LiPo y un GND de la placa | Suena |
| IN− y OUT− del LM2596 | Suena |
| Borne 2 y borne 1 | No suena (si suena, hay un corto) |
| Positivo de la LiPo y GND de la placa | No suena |

## 6. Firmware de prueba: servo y motor

Archivo: `firmware/carro_version_final.ino`. Librería necesaria: **ESP32Servo**.

Qué hace:

1. Al encender, el LED parpadea 10 veces (tiempo para cambiar cables y alimentar).
2. **Prueba del servo:** tres rondas progresivas de ±18°, ±30° y ±46° alrededor del centro (90°), con regreso al centro al final.
3. **Prueba del motor:** sube la potencia de 25 en 25 hasta el tope, la mantiene 8 segundos (tiempo para medir) y baja, primero adelante y luego en reversa.
4. Repite el ciclo.

Decisiones de diseño:

- El servo se inicia **antes** que el motor, igual que en el código original que ya funcionaba, porque la librería ESP32Servo reparte los canales PWM.
- El tope de PWM depende de `VOLTAJE_BATERIA` para proteger el motor de 6 V.
- La zona de medición de 8 segundos permite verificar B+/B−, M+/M− y la LiPo mientras el motor trabaja.

```cpp
// ===== CARRO RC - VERSIÓN FINAL DE PRUEBA =====
// Servo (dirección) en D21 + motor con LiPo + LM2596
// Pines motor: E y F unidos -> D22, G (LPWM) -> D18, H (RPWM) -> D5
// Pines servo: naranja -> D21, rojo -> 5V, café -> GND
//
// Cada ciclo: prueba del servo y luego prueba del motor (adelante y reversa).
// Al encender, el LED parpadea 10 veces antes de empezar.

#include <ESP32Servo.h>

const int LED = 2;

// ===== SERVO - DIRECCIÓN =====
const int PIN_SERVO = 21;
const int CENTRO = 90;                       // ruedas rectas
const int AMPLITUDES[] = {18, 30, 46};       // grados a cada lado del centro
const int NUM_AMPLITUDES = 3;
const int PASO_GRADOS = 1;                   // grados por movimiento
const int RETARDO_SERVO = 25;                // ms entre pasos
const int PAUSA_EXTREMO = 600;               // ms de pausa en cada extremo

Servo direccion;
int posicionActual = CENTRO;

// ===== MOTOR - TRACCIÓN =====
const int EN   = 22;
const int LPWM = 18;
const int RPWM = 5;

// Voltaje que llega a los bornes B+ y B- del BTS7960.
// Con el LM2596 ajustado a 6 V: 6.0
// Con la LiPo directa (sin regulador): 12.6
// Con 2 celdas 18650 en serie: 7.4
const float VOLTAJE_BATERIA = 6.0;
const float VOLTAJE_MOTOR = 6.0;     // RS550 de 6 V
const int VEL_MAX = (VOLTAJE_BATERIA <= VOLTAJE_MOTOR)
                      ? 255
                      : (int)(255.0 * VOLTAJE_MOTOR / VOLTAJE_BATERIA);

const int PASO = 25;                 // igual que PASO_SUBIR del código viejo
const int RETARDO_PASO = 300;        // ms entre pasos
const int TIEMPO_MAXIMO = 8000;      // ms a velocidad tope (tiempo para medir)

// ---------------------------------------------------------------------
// SERVO
// ---------------------------------------------------------------------
void moverSuave(int destino) {
  while (posicionActual != destino) {
    if (posicionActual < destino) {
      posicionActual = min(posicionActual + PASO_GRADOS, destino);
    } else {
      posicionActual = max(posicionActual - PASO_GRADOS, destino);
    }
    direccion.write(posicionActual);
    delay(RETARDO_SERVO);
  }
}

void pruebaServo() {
  Serial.println("=== PRUEBA DE SERVO ===");
  for (int i = 0; i < NUM_AMPLITUDES; i++) {
    int a = AMPLITUDES[i];
    Serial.print("--- Ronda ");
    Serial.print(i + 1);
    Serial.print(": +-");
    Serial.print(a);
    Serial.println(" grados ---");

    Serial.println("DERECHA");
    moverSuave(CENTRO + a);
    delay(PAUSA_EXTREMO);

    Serial.println("CENTRO");
    moverSuave(CENTRO);
    delay(PAUSA_EXTREMO);

    Serial.println("IZQUIERDA");
    moverSuave(CENTRO - a);
    delay(PAUSA_EXTREMO);

    Serial.println("CENTRO");
    moverSuave(CENTRO);
    delay(PAUSA_EXTREMO);
  }

  // Estado final = estado inicial
  direccion.write(CENTRO);
  Serial.println("=== SERVO EN CENTRO ===");
}

// ---------------------------------------------------------------------
// MOTOR
// ---------------------------------------------------------------------
void aplicar(int dir, int pwm) {
  pwm = constrain(pwm, 0, VEL_MAX);
  if (dir == 1) {
    analogWrite(RPWM, pwm);
    analogWrite(LPWM, 0);
  } else {
    analogWrite(LPWM, pwm);
    analogWrite(RPWM, 0);
  }
  Serial.print(dir == 1 ? "ADELANTE" : "ATRAS");
  Serial.print(" PWM: ");
  Serial.print(pwm);
  Serial.print(" (");
  Serial.print((pwm * 100) / 255);
  Serial.println("%)");
}

void detener() {
  analogWrite(LPWM, 0);
  analogWrite(RPWM, 0);
  digitalWrite(LED, LOW);
  Serial.println("MOTOR DETENIDO");
}

void subirYBajar(int dir) {
  digitalWrite(LED, HIGH);
  for (int v = PASO; v < VEL_MAX; v += PASO) {
    aplicar(dir, v);
    delay(RETARDO_PASO);
  }
  aplicar(dir, VEL_MAX);
  Serial.println(">>> TOPE: mide ahora B+/B-, M+/M- y la LiPo (8 s) <<<");
  delay(TIEMPO_MAXIMO);
  for (int v = VEL_MAX - PASO; v > 0; v -= PASO) {
    aplicar(dir, v);
    delay(RETARDO_PASO);
  }
  detener();
}

// ---------------------------------------------------------------------
// GENERAL
// ---------------------------------------------------------------------
void setup() {
  Serial.begin(115200);
  pinMode(LED, OUTPUT);

  // Servo primero (como en el código original que funcionaba)
  direccion.setPeriodHertz(50);
  direccion.attach(PIN_SERVO, 500, 2400);
  direccion.write(CENTRO);
  posicionActual = CENTRO;

  // Motor
  pinMode(EN, OUTPUT);
  pinMode(LPWM, OUTPUT);
  pinMode(RPWM, OUTPUT);
  digitalWrite(EN, HIGH);
  analogWrite(LPWM, 0);
  analogWrite(RPWM, 0);

  Serial.println("=== CARRO RC: SERVO (D21) + MOTOR ===");
  Serial.print("Voltaje en bornes: ");
  Serial.print(VOLTAJE_BATERIA);
  Serial.print(" V | Tope PWM: ");
  Serial.println(VEL_MAX);

  for (int i = 1; i <= 10; i++) {
    digitalWrite(LED, HIGH); delay(500);
    digitalWrite(LED, LOW);  delay(500);
    Serial.print("Parpadeo ");
    Serial.print(i);
    Serial.println(" de 10");
  }
}

void loop() {
  // Prueba 1: servo
  pruebaServo();
  delay(2000);

  // Prueba 2: motor
  Serial.println("##### ADELANTE #####");
  subirYBajar(1);
  delay(2000);

  Serial.println("##### REVERSA #####");
  subirYBajar(-1);
  delay(5000);
}

```

## 7. Firmware con control Xbox

Archivo: `firmware/carro_xbox_bluepad32.ino`. Librerías: **Bluepad32** (incluida en la placa `esp32_bluepad32`) y **ESP32Servo**. Placa seleccionada en el IDE: ESP32 Dev Module de `esp32_bluepad32`.

### Funcionamiento

| Control | Acción |
|---|---|
| Joystick derecho, horizontal | Dirección: de 44° a 136° (90° ± 46°, el límite calibrado) |
| Joystick izquierdo, vertical | Motor: arriba = adelante, abajo = reversa. Más lejos del centro = más potencia |

### Características

- **Control proporcional** con zona muerta (`ZONA_MUERTA = 60`) para ignorar el movimiento pequeño cerca del centro.
- **Rampas de velocidad** (`RAMPA_SUBIDA`, `RAMPA_BAJADA`) para suavizar el arranque y no exigir picos de corriente al LM2596.
- **Suavizado del servo**: filtro exponencial más límite de velocidad, para evitar cambios de sentido bruscos.
- **Motor armado**: no responde hasta que el joystick izquierdo pase por el centro después de conectar el control.
- **Failsafe**: si el control se desconecta, el motor se detiene y la dirección vuelve al centro.
- **LED**: parpadea mientras no hay control y queda fijo cuando está conectado.
- **Monitor serial**: muestra el modelo, VID y PID del control al conectar; los errores; el motivo de reinicio de la ESP32; y los movimientos solo cuando hay movimiento.
- Acepta el control aunque Bluepad32 no lo clasifique como gamepad, siempre que envíe datos.

### Parámetros ajustables

| Constante | Uso |
|---|---|
| `INVERTIR_DIRECCION` | Cambiar si la dirección gira al lado contrario del joystick |
| `LIMITE_GIRO` | Grados máximos a cada lado (46 calibrado) |
| `SERVO_PASO_MAX` y `SERVO_SUAVIZADO` | Velocidad y suavidad del servo |
| `ZONA_MUERTA` | Sensibilidad cerca del centro |
| `PWM_MINIMO` | PWM al que el motor empieza a girar |
| `RAMPA_SUBIDA` y `RAMPA_BAJADA` | Aceleración y frenado |
| `OLVIDAR_PAREOS` | Borra los emparejamientos guardados (usar una vez si no empareja) |
| `FILTRAR_POR_ID` | Acepta solo un control con el VID y PID indicados |
| `DESACTIVAR_BROWNOUT` | Solo diagnóstico, no dejar activado |

### Emparejamiento

1. Encender el control con el botón Xbox.
2. Mantener presionado el botón de emparejamiento de arriba hasta que el logo parpadee rápido.
3. La primera vez se empareja; después se reconecta solo al encenderlo.
4. Si el control estaba emparejado con otro dispositivo, conviene desvincularlo primero.

```cpp
// ===== CARRO RC CON CONTROL DE XBOX (Bluepad32) - v2 =====
// Joystick DERECHO (horizontal) -> dirección (servo en D21)
// Joystick IZQUIERDO (vertical) -> potencia del motor, proporcional
//     arriba = adelante | abajo = reversa | más lejos del centro = más potencia
//
// Pines motor: E y F unidos -> D22, G (LPWM) -> D18, H (RPWM) -> D5
// Pines servo: naranja -> D21, rojo -> 5V, café -> GND
// Placa en el IDE: ESP32 Dev Module (esp32_bluepad32). Librería: ESP32Servo.
//
// LED: parpadea mientras no hay control, fijo cuando está conectado.
// Monitor serial: muestra errores y datos del control al conectar.
// Los movimientos solo se imprimen cuando hay movimiento.

#include <Bluepad32.h>
#include <ESP32Servo.h>
#include <esp_system.h>
#include "soc/soc.h"
#include "soc/rtc_cntl_reg.h"

const int LED = 2;

// ===== SERVO - DIRECCIÓN =====
const int PIN_SERVO = 21;
const int CENTRO = 90;                 // ruedas rectas
const int LIMITE_GIRO = 46;            // grados máximos a cada lado (calibrado)
const bool INVERTIR_DIRECCION = false; // true si gira al revés
const float SERVO_PASO_MAX = 1.5;      // grados máximos por ciclo de 10 ms (1.5 = 150 grados/s)
const float SERVO_SUAVIZADO = 0.2;     // 0.05 = muy suave | 1.0 = sin suavizado
// SOLO PARA DIAGNOSTICO: si al desactivarlo ya no se desconecta, el problema es voltaje.
const bool DESACTIVAR_BROWNOUT = false;

Servo direccion;
int anguloActual = CENTRO;
float anguloF = CENTRO;                // posición suavizada (decimal)

// ===== MOTOR - TRACCIÓN =====
const int EN   = 22;
const int LPWM = 18;
const int RPWM = 5;

// Voltaje en los bornes B+ y B-: LM2596 a 6 V = 6.0 | LiPo directa = 12.6
const float VOLTAJE_BATERIA = 6.0;
const float VOLTAJE_MOTOR = 6.0;       // RS550 de 6 V
const int VEL_MAX = (VOLTAJE_BATERIA <= VOLTAJE_MOTOR)
                      ? 255
                      : (int)(255.0 * VOLTAJE_MOTOR / VOLTAJE_BATERIA);

const int PWM_MINIMO = 0;              // sube si el motor no arranca con poco joystick
const int RAMPA_SUBIDA = 6;            // cambio máximo de PWM por ciclo al acelerar
const int RAMPA_BAJADA = 12;           // cambio máximo de PWM por ciclo al frenar

int velocidadActual = 0;               // de -VEL_MAX (reversa) a +VEL_MAX (adelante)

// ===== JOYSTICK =====
const int EJE_MAX = 512;               // Bluepad32 entrega de -511 a 512
const int ZONA_MUERTA = 60;            // ignora movimiento pequeño cerca del centro

// ===== CONEXIÓN DEL CONTROL =====
const bool OLVIDAR_PAREOS = false;     // true una vez si no logra emparejar
const bool FILTRAR_POR_ID = false;     // true para aceptar solo tu control
const uint16_t VID_PERMITIDO = 0x0000; // pon aquí el VID que muestra el monitor
const uint16_t PID_PERMITIDO = 0x0000; // pon aquí el PID que muestra el monitor

ControllerPtr control = nullptr;
bool armado = false;
bool avisoArmado = false;
bool avisoNoGamepad = false;
bool avisoSinClase = false;

unsigned long ultimoCiclo = 0;
unsigned long ultimoMensaje = 0;
unsigned long ultimoParpadeo = 0;
int ultimoAngulo = CENTRO;
int ultimoPwm = 0;

// ---------------------------------------------------------------------
// CALLBACKS DE BLUEPAD32
// ---------------------------------------------------------------------
void onConnectedController(ControllerPtr ctl) {
  ControllerProperties p = ctl->getProperties();

  if (control != nullptr) {
    Serial.println("ERROR: ya hay un control conectado. Se ignora el nuevo.");
    return;
  }
  if (FILTRAR_POR_ID && (p.vendor_id != VID_PERMITIDO || p.product_id != PID_PERMITIDO)) {
    Serial.printf("ERROR: control rechazado por ID (VID=0x%04X, PID=0x%04X).\n",
                  p.vendor_id, p.product_id);
    return;
  }

  control = ctl;
  armado = false;
  avisoArmado = false;
  avisoNoGamepad = false;
  avisoSinClase = false;
  Serial.printf("Control conectado: %s | VID=0x%04X | PID=0x%04X | gamepad: %s\n",
                ctl->getModelName().c_str(), p.vendor_id, p.product_id,
                ctl->isGamepad() ? "si" : "aun no");
}

void onDisconnectedController(ControllerPtr ctl) {
  if (control == ctl) {
    control = nullptr;
    Serial.println("AVISO: control desconectado. Motor detenido y direccion al centro.");
  }
}

// ---------------------------------------------------------------------
// FUNCIONES
// ---------------------------------------------------------------------
// Convierte el valor del eje en un número de -1.0 a 1.0, con zona muerta
float normalizar(int valor) {
  valor = constrain(valor, -EJE_MAX, EJE_MAX);
  int magnitud = abs(valor);
  if (magnitud <= ZONA_MUERTA) return 0.0;
  float n = (float)(magnitud - ZONA_MUERTA) / (EJE_MAX - ZONA_MUERTA);
  return (valor < 0) ? -n : n;
}

void aplicarMotor(int v) {
  if (v > 0) {
    analogWrite(RPWM, v);
    analogWrite(LPWM, 0);
  } else if (v < 0) {
    analogWrite(LPWM, -v);
    analogWrite(RPWM, 0);
  } else {
    analogWrite(LPWM, 0);
    analogWrite(RPWM, 0);
  }
}

// Informa por qué arrancó la ESP32 (sirve para detectar caídas de voltaje)
void imprimirMotivoReinicio() {
  esp_reset_reason_t r = esp_reset_reason();
  switch (r) {
    case ESP_RST_POWERON:
      Serial.println("Arranque: encendido normal.");
      break;
    case ESP_RST_BROWNOUT:
      Serial.println("ERROR: reinicio por BROWNOUT (caida de voltaje en la alimentacion).");
      break;
    case ESP_RST_PANIC:
      Serial.println("ERROR: reinicio por fallo del programa (panic).");
      break;
    case ESP_RST_INT_WDT:
    case ESP_RST_TASK_WDT:
    case ESP_RST_WDT:
      Serial.println("ERROR: reinicio por watchdog (el programa se bloqueo).");
      break;
    case ESP_RST_SW:
      Serial.println("Arranque: reinicio por software.");
      break;
    case ESP_RST_EXT:
      Serial.println("Arranque: reinicio por boton EN.");
      break;
    default:
      Serial.printf("Arranque: motivo de reinicio %d\n", (int)r);
  }
}

// ---------------------------------------------------------------------
// SETUP
// ---------------------------------------------------------------------
void setup() {
  if (DESACTIVAR_BROWNOUT) {
    WRITE_PERI_REG(RTC_CNTL_BROWN_OUT_REG, 0);   // desactiva el detector de caída de voltaje
  }
  Serial.begin(115200);
  delay(500);
  if (DESACTIVAR_BROWNOUT) {
    Serial.println("AVISO: detector de brownout DESACTIVADO (solo para diagnostico).");
  }
  imprimirMotivoReinicio();
  pinMode(LED, OUTPUT);

  // Servo primero (como en el código que ya funcionaba)
  direccion.setPeriodHertz(50);
  direccion.attach(PIN_SERVO, 500, 2400);
  if (!direccion.attached()) {
    Serial.println("ERROR: no se pudo iniciar el servo en D21.");
  }
  direccion.write(CENTRO);
  anguloActual = CENTRO;
  anguloF = CENTRO;

  // Motor
  pinMode(EN, OUTPUT);
  pinMode(LPWM, OUTPUT);
  pinMode(RPWM, OUTPUT);
  digitalWrite(EN, HIGH);
  aplicarMotor(0);

  // Bluepad32
  BP32.setup(&onConnectedController, &onDisconnectedController);
  if (OLVIDAR_PAREOS) {
    BP32.forgetBluetoothKeys();
    Serial.println("AVISO: se borraron los emparejamientos guardados.");
  }

  Serial.printf("Listo. Tope PWM: %d | Giro maximo: +-%d grados\n", VEL_MAX, LIMITE_GIRO);
  Serial.println("Esperando control...");
}

// ---------------------------------------------------------------------
// LOOP
// ---------------------------------------------------------------------
void loop() {
  BP32.update();

  unsigned long ahora = millis();
  if (ahora - ultimoCiclo < 10) {         // un ciclo cada 10 ms
    delay(1);                             // cede tiempo al Bluetooth
    return;
  }
  ultimoCiclo = ahora;

  int objetivoVelocidad = 0;
  int objetivoAngulo = CENTRO;

  bool conectado = (control != nullptr && control->isConnected());
  bool esGamepad = conectado && control->isGamepad();
  bool listo = conectado && (esGamepad || control->hasData());

  if (conectado && !listo && !avisoNoGamepad) {
    avisoNoGamepad = true;
    Serial.println("ERROR: control conectado pero sin datos. Bluepad32 no lo reconoce.");
  }
  if (listo && !esGamepad && !avisoSinClase) {
    avisoSinClase = true;
    Serial.println("AVISO: Bluepad32 no lo clasifica como gamepad, pero envia datos. Se usan los ejes.");
  }

  if (listo) {
    digitalWrite(LED, HIGH);              // fijo: control conectado y con datos
    if (avisoNoGamepad) {
      avisoNoGamepad = false;
      Serial.println("Control listo.");
    }

    // --- Dirección (joystick derecho) ---
    float giro = normalizar(control->axisRX());
    if (INVERTIR_DIRECCION) giro = -giro;
    objetivoAngulo = CENTRO + (int)round(giro * LIMITE_GIRO);

    // --- Motor (joystick izquierdo, arriba = adelante) ---
    float potencia = -normalizar(control->axisY());

    if (!armado) {
      if (potencia == 0.0) {
        armado = true;
        Serial.println("Motor armado.");
      } else if (!avisoArmado) {
        avisoArmado = true;
        Serial.println("AVISO: lleva el joystick izquierdo al centro para armar el motor.");
      }
    } else if (potencia != 0.0) {
      int magnitud = PWM_MINIMO + (int)(fabs(potencia) * (VEL_MAX - PWM_MINIMO));
      magnitud = constrain(magnitud, 0, VEL_MAX);
      objetivoVelocidad = (potencia > 0) ? magnitud : -magnitud;
    }
  } else {
    // Sin control: parar el motor, centrar la dirección y parpadear el LED
    velocidadActual = 0;
    armado = false;
    if (ahora - ultimoParpadeo >= 500) {
      ultimoParpadeo = ahora;
      digitalWrite(LED, !digitalRead(LED));
    }
  }

  // --- Rampa de velocidad (protege el regulador y suaviza el arranque) ---
  bool acelerando = (velocidadActual == 0) ||
                    ((long)objetivoVelocidad * velocidadActual > 0 &&
                     abs(objetivoVelocidad) > abs(velocidadActual));
  int paso = acelerando ? RAMPA_SUBIDA : RAMPA_BAJADA;

  if (velocidadActual < objetivoVelocidad) {
    velocidadActual = min(velocidadActual + paso, objetivoVelocidad);
  } else if (velocidadActual > objetivoVelocidad) {
    velocidadActual = max(velocidadActual - paso, objetivoVelocidad);
  }
  aplicarMotor(velocidadActual);

  // --- Servo: suavizado exponencial + límite de velocidad ---
  // Evita arranques y cambios de sentido bruscos, que son los que piden más corriente.
  float diferencia = (objetivoAngulo - anguloF) * SERVO_SUAVIZADO;
  diferencia = constrain(diferencia, -SERVO_PASO_MAX, SERVO_PASO_MAX);
  anguloF += diferencia;
  anguloF = constrain(anguloF, CENTRO - LIMITE_GIRO, CENTRO + LIMITE_GIRO);

  int nuevoAngulo = (int)round(anguloF);
  if (nuevoAngulo != anguloActual) {       // solo se manda al servo si cambió
    anguloActual = nuevoAngulo;
    direccion.write(anguloActual);
  }

  // --- Monitor: solo si hay movimiento y cambió algo ---
  bool enMovimiento = (anguloActual != CENTRO) || (velocidadActual != 0);
  if (enMovimiento) {
    if ((anguloActual != ultimoAngulo || velocidadActual != ultimoPwm) &&
        ahora - ultimoMensaje >= 200) {
      ultimoMensaje = ahora;
      ultimoAngulo = anguloActual;
      ultimoPwm = velocidadActual;
      Serial.printf("Dir: %d | PWM: %d (%d%%)\n",
                    anguloActual, velocidadActual, (abs(velocidadActual) * 100) / 255);
    }
  } else {
    ultimoAngulo = CENTRO;
    ultimoPwm = 0;
  }
}

```

## 8. Procedimiento de puesta en marcha

1. Dejar todas las baterías desconectadas y cargar el firmware por USB.
2. Desconectar el USB.
3. Conectar el pack de 3 celdas al puerto DC de la placa (SW1).
4. Encender la LiPo (SW2).
5. Con el firmware de prueba: el LED parpadea 10 veces y empieza la rutina. Con el firmware Xbox: encender el control y esperar a que el LED quede fijo.
6. Probar siempre con las ruedas en el aire.

### Revisión antes de la prueba (todo apagado)

Con continuidad: ver la tabla de la sección 5. Con voltaje:

| Medición | Valor esperado |
|---|---|
| LiPo | Entre 11 y 12.6 V |
| OUT+ y OUT− del LM2596 sin el BTS7960 | 6.0 V |
| Pin A y pin B del BTS7960, con la placa encendida | 5 V |

### Revisión durante la prueba

| Medición | Valor esperado |
|---|---|
| Borne 2 (B+) respecto al borne 1 | Entre 5.5 y 6.2 V, estable |
| Pines E y F respecto al pin A | 3.3 V |
| Pin H (adelante) o G (reversa) | Voltaje intermedio que sube con el PWM |
| LiPo con carga | Más de 10 V |
| Temperatura del LM2596 | Tibio. Si quema al tacto, apagar |

### Si el motor no gira

| Lo que se ve | Interpretación |
|---|---|
| B+ cae por debajo de 5.5 V al arrancar | El LM2596 no aguanta la corriente. Alternativa: LiPo directa al driver con `VOLTAJE_BATERIA = 12.6` |
| B+ da 0 V desde el inicio | Falla antes del borne: interruptor, fusible, IN+/IN− o regulador sin calibrar |
| B+ bien y M+ da 0 V | El BTS7960 no activa: revisar pin B (5 V), E y F (3.3 V) y el pin A a GND |
| M+ tiene voltaje y el motor no gira | Cables o tornillos de los bornes 3 y 4 sin contacto |

## 9. Estado actual y problema abierto

**Funciona:**

- Motor en adelante y reversa con la LiPo, el LM2596 a 6 V y el BTS7960.
- Servo en D21 con la rutina progresiva de ±18°, ±30° y ±46°.
- El control Xbox se empareja. El monitor muestra el modelo (XBox One, VID 0x045E, PID 0x02FD) y los valores de los joysticks se convierten correctamente en ángulo y PWM.
- Con el control, el motor responde bien.

**Problema abierto:** con el control Xbox, al mover la dirección (derecha e izquierda) el control se desconecta. El motor por sí solo no provoca la desconexión. Se probó alimentando solo con el pack de 3 celdas y la LiPo, sin USB.

Hipótesis principal (**no confirmada**): el servo MG995 provoca un pico de corriente al cambiar de sentido, la alimentación de la placa cae y la ESP32 se reinicia, lo que corta el Bluetooth. Un indicio: en una captura, el monitor mostró "Not connected" (el puerto USB desapareció), algo que no ocurre por un fallo de Bluetooth.

**Acciones propuestas, pendientes de validar:**

| Acción | Detalle |
|---|---|
| Capacitor | De 470 a 1000 µF (16 V o más) entre el rojo y el café del servo, con la franja al GND |
| Alimentación aparte del servo | Rojo a un segundo regulador de 5 V desde el pack de 3 celdas (o, como prueba, a los 6 V del LM2596), con GND común |
| Suavizado en código | Ya aplicado: límite de velocidad y filtro exponencial en el servo |
| Diagnóstico | El monitor imprime el motivo de reinicio (por ejemplo, BROWNOUT). La opción `DESACTIVAR_BROWNOUT` sirve solo para confirmar la causa |
| Límite de giro | Probar `LIMITE_GIRO = 40` si el servo topa mecánicamente con carga en el piso |

Si el monitor no muestra ningún reinicio y el control se corta de todos modos, la causa no sería la alimentación y habría que investigar el Bluetooth.

## 10. Historial de pruebas y lecciones aprendidas

| Síntoma | Causa encontrada |
|---|---|
| El motor no giraba con una celda de 3.7 V | El BTS7960 necesita al menos 5.5 V en B+/B−. Con 2 celdas (7.4 V) giró |
| VCC del driver sin 5 V | El jumper de la placa estaba en una posición incorrecta. Debe cubrir 5V y V |
| El motor no giraba con el firmware de prueba en D25, D26, D27 y D33 | Con señal digital fija sí giraba, con PWM no. **No se determinó la causa.** Con D22, D18 y D5 funciona |
| El pin 12 de la ESP32 | Es un pin de arranque. Se evitó |
| Humo al conectar la batería por primera vez | Evento aislado en el primer montaje. La continuidad no mostró cortos y el módulo siguió funcionando |
| La batería caía de 3.7 V a 1.7 V al activar el driver | Aparecía con una sola celda. Con 2 celdas los voltajes fueron los esperados |
| El LED del regulador seguía encendido con el circuito apagado | El interruptor estaba después de OUT+. Se movió antes de IN+ |
| Error de compilación `ESP32Servo.h: No such file` | Faltaba instalar la librería ESP32Servo |
| Error "Could not open COM3, the port doesn't exist" | El puerto USB se desconectó un momento. Se reconectó el cable |
| El control conectaba pero el LED seguía parpadeando | El código rechazaba el control al conectar porque `isGamepad()` aún era falso. Ahora lo acepta si envía datos |
| El control Xbox se desconecta al mover la dirección | Problema abierto, ver sección 9 |

### Orden de diagnóstico que funcionó para el BTS7960

1. Medir 5 V entre A y B.
2. Activar E, F y H a mano con 5 V y comprobar que la salida M+ da voltaje.
3. Probar con una batería con voltaje suficiente (7.4 V) antes de culpar al módulo.
4. Pasar al control por la ESP32 y comparar con la prueba manual.

## 11. Seguridad

- Verificar la polaridad con el multímetro antes de conectar cualquier batería, sin confiar solo en el color de los cables.
- Usar fusible de 10 a 15 A en el positivo de la LiPo.
- No cortocircuitar celdas 18650 sin protección.
- Evitar pasar de 6 V efectivos en el RS550: limitar el PWM según el voltaje de la fuente.
- No dejar la LiPo conectada sin supervisión ni descargarla por debajo de unos 9 V.
- Desconectar la batería del motor antes de mover cables.
- Probar siempre con las ruedas en el aire.

## 12. Pendientes

- [ ] Resolver la desconexión del control al mover la dirección y confirmar la causa.
- [ ] Validar el LM2596 con el motor bajo carga (límite de unos 3 A).
- [ ] Definir si el servo se alimenta desde un regulador de 5 V propio.
- [ ] Calibrar la dirección con el control: precisión y límites de recorrido en el piso.
- [ ] Ajustar `PWM_MINIMO` midiendo el PWM al que arranca el motor.
- [ ] Documentar tiempos de vuelta y consumo en pista.
- [ ] Completar las imágenes de este documento.

---

*Documento generado a partir del proceso de pruebas del proyecto. Las imágenes referenciadas deben guardarse en la carpeta `img/` con los nombres indicados.*